In [13]:
import numpy as np

# =====================================================================
# FUNDAMENTAL CONSTANTS (CGS SYSTEM)
# =====================================================================
C_LIGHT  = 2.99792458e10  # Speed of light [cm/s]
K_B      = 1.380649e-16   # Boltzmann constant [erg/K]
M_P      = 1.6726219e-24  # Proton mass [g]
G        = 6.67430e-8     # Gravitational constant [cm^3 / (g * s^2)]

# Astronomical units for conversion
M_SUN    = 1.9884e33      # Solar mass [g]
PC       = 3.08567758e18  # Parsec [cm]
SEC_IN_YR = 3.15576e7     # Seconds in a year

In [14]:
# =====================================================================
# MODEL INPUT PARAMETERS
# =====================================================================

# 1. Thermodynamics and scaling
T_0      = 1e4            # Reference temperature [K]
mu       = 2.3            # Mean molecular weight
gamma    = 5.0 / 3.0      # Adiabatic index
chi      = 6.98e5         # Spatial scaling parameter (r_0 / r_g)
alpha    = 0.01           # Shakura-Sunyaev viscosity parameter

# 2. Athena++ grid geometry
r_center = 1.0            # Dimensionless disk center offset (R_0 / r_0)
C_prime  = 0.1            # Geometric disk thickness parameter

# 3. Simulation settings
N_orbits = 100.0          # Number of orbital periods to simulate
frames_per_orbit = 10.0   # Number of output frames per orbit

# 4. Object physics (For post-processing)
M_BH_sun = 1.5e7          # Black hole mass in solar masses

In [15]:
# =====================================================================
# DIMENSIONLESS PARAMETERS FOR ATHENA++ (PRE-PROCESSING)
# =====================================================================

# 1. Physical sound speed (v_0)
cs0 = np.sqrt(gamma * K_B * T_0 / (mu * M_P))

# 2. Dimensionless gravity (beta)
beta = C_LIGHT**2 / (2.0 * chi * cs0**2)

# 3. Dimensionless time for one orbital period at the disk center
P_tilde = 2.0 * np.pi * np.sqrt(r_center**3 / beta)

# 4. Viscous scaling
n_poly = 1.0 / (gamma - 1.0)
# Dimensionless central kinematic viscosity (nu_iso)
nu_0 = alpha * gamma * np.sqrt(beta) * ((0.5 - C_prime) / (n_poly + 1.0)) * np.sqrt(r_center)
# Dimensionless viscous time scale
tau_visc_tilde = r_center**2 / nu_0

# 5. Time parameters for athinput
dt_out = P_tilde / frames_per_orbit
tlim = P_tilde * N_orbits

# =====================================================================
# PHYSICAL SCALES (POST-PROCESSING)
# =====================================================================

M_BH = M_BH_sun * M_SUN
r_g = 2.0 * G * M_BH / C_LIGHT**2  # Gravitational radius [cm]

# Scales for decoding results
r_0 = chi * r_g                    # Length scale [cm]
t_0 = r_0 / cs0                    # Time scale [s]
p_0 = cs0**2                       # Kinematic pressure scale [cm^2/s^2]

# Physical times
P_phys_sec = P_tilde * t_0
P_phys_yr = P_phys_sec / SEC_IN_YR

tau_visc_phys_sec = tau_visc_tilde * t_0
tau_visc_phys_yr = tau_visc_phys_sec / SEC_IN_YR

In [16]:
# =====================================================================
# FORMATTED OUTPUT
# =====================================================================

print("="*65)
print(" 1. PARAMETERS FOR ATHINPUT (Dimensionless Units)")
print("="*65)
print("<output1>")
print(f"dt         = {dt_out:.5e}  # Save frame every {1/frames_per_orbit} orbits")
print("\n<time>")
print(f"tlim       = {tlim:.5e}  # Time limit for {N_orbits} full orbits")
print("\n<hydro>")
print(f"nu_iso     = {nu_0:.5e}  # Central kinematic viscosity (alpha = {alpha})")
print("\n<problem>")
print(f"r_center   = {r_center}")
print(f"C_prime    = {C_prime}")
print(f"T_0        = {T_0:.2e}")
print(f"mu         = {mu}")
print(f"chi        = {chi:.2e}")
print("-" * 65)
print(f"[INFO] Calculated gravity    : beta           = {beta:.3f}")
print(f"[INFO] 1 Orbit period        : P_tilde        = {P_tilde:.5f}")
print(f"[INFO] Viscous time scale    : tau_visc_tilde = {tau_visc_tilde:.5f}")

print("\n" + "="*65)
print(" 2. SCALES FOR POST-PROCESSING (Physical Units)")
print("="*65)
print(f"Velocity scale (v_0 = cs0) : {cs0/1e5:.3f} km/s")
print(f"Length scale (r_0)         : {r_0:.3e} cm ({r_0/PC:.3f} pc)")
print(f"Time scale (t_0)           : {t_0:.3e} s ({t_0/SEC_IN_YR:.3e} yr)")
print("-" * 65)
print(f"Physical 1 orbit time      : {P_phys_yr:.1f} yr")
print(f"Total simulation time      : {P_phys_yr * N_orbits:.1f} yr")
print(f"Physical viscous time      : {tau_visc_phys_yr:.1e} yr")
print(f"Ratio (tau_visc / P_orbit) : {tau_visc_tilde / P_tilde:.1f} orbits")
print("="*65)

 1. PARAMETERS FOR ATHINPUT (Dimensionless Units)
<output1>
dt         = 1.91516e-02  # Save frame every 0.1 orbits

<time>
tlim       = 1.91516e+01  # Time limit for 100.0 full orbits

<hydro>
nu_iso     = 8.74870e-02  # Central kinematic viscosity (alpha = 0.01)

<problem>
r_center   = 1.0
C_prime    = 0.1
T_0        = 1.00e+04
mu         = 2.3
chi        = 6.98e+05
-----------------------------------------------------------------
[INFO] Calculated gravity    : beta           = 1076.340
[INFO] 1 Orbit period        : P_tilde        = 0.19152
[INFO] Viscous time scale    : tau_visc_tilde = 11.43027

 2. SCALES FOR POST-PROCESSING (Physical Units)
Velocity scale (v_0 = cs0) : 7.734 km/s
Length scale (r_0)         : 3.092e+18 cm (1.002 pc)
Time scale (t_0)           : 3.998e+12 s (1.267e+05 yr)
-----------------------------------------------------------------
Physical 1 orbit time      : 24262.9 yr
Total simulation time      : 2426289.7 yr
Physical viscous time      : 1.4e+06 yr
Ratio (